# 🏛️ Nano ChessGPT — 85M
> Character-level chess language model · 26M Lichess Elite games · GPT-2 Small

| | |
|--|--|
| **Params** | 84.98M |
| **Architecture** | 12L × 12H × 768D |
| **Dataset** | 9.19B tokens (Lichess Elite DB) |
| **Vocab** | 29 chars (character-level) |
| **Target** | ~1500–1800 ELO |

---
### 📋 প্রতি Session এ কী করবেন:
```
Cell 1 → GPU check         (5 sec)
Cell 2 → Drive + Dataset   (10-15 min, Drive থেকে copy)
Cell 3 → Repo clone        (30 sec)
Cell 4 → Training          (চলতে থাকবে, auto-resume)
Cell 5 → Checkpoint save   (session শেষ হওয়ার আগে!)
```
> **Session disconnect?** → Cell 1→2→3→4 চালান, auto-resume হবে ✅

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 1 ── GPU & Environment Check
# ═══════════════════════════════════════════════════
import torch, subprocess, psutil

print('=' * 52)
print('  Nano ChessGPT 85M ── Environment Check')
print('=' * 52)

# GPU
assert torch.cuda.is_available(), (
    '❌ GPU নেই! Runtime → Change runtime type → T4 GPU'
)
gpu   = torch.cuda.get_device_name(0)
vram  = torch.cuda.get_device_properties(0).total_memory / 1024**3
bf16  = torch.cuda.is_bf16_supported()
arch  = torch.cuda.get_device_capability()

# T4 = sm_75. float16 native. bfloat16 = software only → SLOW!
# A100/H100 = sm_80+. bfloat16 native.
USE_FLOAT16 = arch[0] < 8   # sm < 80 → use float16
dtype_str   = 'float16' if USE_FLOAT16 else 'bfloat16'

print(f'\n✅ GPU  : {gpu}')
print(f'   VRAM : {vram:.1f} GB')
print(f'   Arch : sm_{arch[0]}{arch[1]} ({"Turing" if arch[0]==7 else "Ampere+" if arch[0]>=8 else "Other"})')
print(f'   dtype: {dtype_str} ← {"T4 native Tensor Core" if USE_FLOAT16 else "bfloat16 native"} ✅')
print(f'   PyTorch : {torch.__version__} | CUDA: {torch.version.cuda}')

# Disk & RAM
r   = subprocess.run(['df','-h','/'], capture_output=True, text=True)
d   = r.stdout.strip().split('\n')[1].split()
ram = psutil.virtual_memory()
print(f'\n💾 Disk : {d[3]} free / {d[1]} total')
print(f'   RAM  : {ram.available/1024**3:.1f} GB free / {ram.total/1024**3:.1f} GB')

if vram < 14:
    print('\n⚠️  VRAM < 14 GB — consider reducing batch_size')

# Save dtype for later cells
import builtins
builtins.CHESS_DTYPE = dtype_str
print(f'\n✅ Ready! dtype={dtype_str}')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 2 ── Mount Drive + Copy Dataset
# Drive → Colab local disk (~10-15 min)
# ═══════════════════════════════════════════════════
from google.colab import drive
import os, shutil, time

DRIVE_DATA = '/content/drive/MyDrive/NanoChessGPT'
DRIVE_CKPT = '/content/drive/MyDrive/NanoChessGPT/checkpoints'
LOCAL_DATA = '/content/chess_data'

drive.mount('/content/drive')
os.makedirs(LOCAL_DATA, exist_ok=True)
os.makedirs(DRIVE_CKPT, exist_ok=True)
print('✅ Drive mounted!')

# ── Check Drive files ────────────────────────────
files = ['train.bin', 'val.bin', 'meta.pkl']
print(f'\n📁 Drive: {DRIVE_DATA}/')
all_ok = True
for f in files:
    p = f'{DRIVE_DATA}/{f}'
    if os.path.exists(p):
        print(f'  ✅ {f:12s} ({os.path.getsize(p)/1024**3:.2f} GB)')
    else:
        print(f'  ❌ {f} — NOT FOUND!')
        all_ok = False
if not all_ok:
    raise FileNotFoundError('Drive-এ dataset নেই! আগে upload করুন।')

# ── Copy to local (skip if already exists) ───────
print(f'\n📦 Copying to local disk...')
for f in files:
    src = f'{DRIVE_DATA}/{f}'
    dst = f'{LOCAL_DATA}/{f}'
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        print(f'  ✅ {f:12s} already local — skip')
        continue
    sz  = os.path.getsize(src) / 1024**3
    print(f'  ⏳ {f} ({sz:.2f} GB)...', end='', flush=True)
    t0  = time.time()
    shutil.copy2(src, dst)
    dt  = time.time() - t0
    print(f' {dt/60:.1f} min ({sz/dt*1024:.0f} MB/s) ✅')

# ── Existing checkpoints ─────────────────────────
ckpts = sorted([f for f in os.listdir(DRIVE_CKPT) if f.endswith('.pt')])
print(f'\n🔄 Checkpoints in Drive: {len(ckpts)}')
for c in ckpts:
    sz = os.path.getsize(f'{DRIVE_CKPT}/{c}') / 1024**2
    print(f'   • {c} ({sz:.0f} MB)')
print(f'\n✅ Dataset ready → Run Cell 3')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 3 ── Clone Repo & Link Dataset
# ═══════════════════════════════════════════════════
import os, shutil

REPO_URL   = 'https://github.com/ItsMohammadSajid/NanoChessGPT-85M.git'
REPO_DIR   = '/content/NanoChessGPT-85M'
LOCAL_DATA = '/content/chess_data'
DRIVE_CKPT = '/content/drive/MyDrive/NanoChessGPT/checkpoints'
LOCAL_OUT  = f'{REPO_DIR}/out-chess-gpt2small'
CHESS_DIR  = f'{REPO_DIR}/data/chess'

# Clone
if os.path.exists(REPO_DIR):
    print(f'✅ Repo already at {REPO_DIR}')
else:
    print('📦 Cloning repo...')
    !git clone --depth=1 {REPO_URL} {REPO_DIR}
    print('✅ Cloned!')

os.chdir(REPO_DIR)
os.makedirs(LOCAL_OUT, exist_ok=True)
os.makedirs(CHESS_DIR, exist_ok=True)

# Symlink dataset
for f in ['train.bin', 'val.bin', 'meta.pkl']:
    src = f'{LOCAL_DATA}/{f}'
    dst = f'{CHESS_DIR}/{f}'
    if os.path.islink(dst) or os.path.exists(dst):
        os.remove(dst) if os.path.islink(dst) else None
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'  🔗 {f}')
    elif os.path.exists(dst):
        print(f'  ✅ {f}')
    else:
        print(f'  ❌ {f} missing! Run Cell 2.')

# Copy Drive checkpoints → local
ckpts = sorted([f for f in os.listdir(DRIVE_CKPT) if f.endswith('.pt')]) if os.path.exists(DRIVE_CKPT) else []
if ckpts:
    print(f'\n🔄 Copying {len(ckpts)} checkpoint(s) from Drive...')
    for c in ckpts:
        src = f'{DRIVE_CKPT}/{c}'
        dst = f'{LOCAL_OUT}/{c}'
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
        sz = os.path.getsize(dst)/1024**2
        print(f'   ✅ {c} ({sz:.0f} MB)')
else:
    print(f'\n🆕 No checkpoints — fresh training')

print(f'\n✅ Setup complete!')
!ls -lh data/chess/

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 4 ── TRAINING
# • float16  → T4 Tensor Core (native, fast!)
# • compile  → torch.compile  (~20-30% speedup)
# • Auto-detects fresh start vs resume
# • Auto-saves checkpoint to Drive every 30 min
# ═══════════════════════════════════════════════════
import os, glob, shutil, threading, time

REPO_DIR   = '/content/NanoChessGPT-85M'
LOCAL_OUT  = f'{REPO_DIR}/out-chess-gpt2small'
DRIVE_CKPT = '/content/drive/MyDrive/NanoChessGPT/checkpoints'
TRAIN_BIN  = f'{REPO_DIR}/data/chess/train.bin'

os.chdir(REPO_DIR)

# ── Verify dataset ───────────────────────────────
assert os.path.exists(TRAIN_BIN), '❌ train.bin নেই! Cell 2 & 3 আগে চালান।'
sz = os.path.getsize(TRAIN_BIN) / 1024**3
print(f'✅ train.bin ({sz:.1f} GB)')

# ── Detect fresh vs resume ───────────────────────
existing  = sorted(glob.glob(f'{LOCAL_OUT}/*.pt'))
INIT_FROM = 'resume' if existing else 'scratch'
if existing:
    print(f'🔄 RESUMING from: {os.path.basename(existing[-1])}')
else:
    print(f'🆕 Fresh training')

# ── Auto dtype from Cell 1 ───────────────────────
import builtins
dtype = getattr(builtins, 'CHESS_DTYPE', 'float16')
print(f'⚡ dtype: {dtype} | compile: True')

# ── Auto-save to Drive (background thread) ────────
def auto_save_thread():
    interval = 30 * 60  # every 30 minutes
    time.sleep(interval)
    while True:
        try:
            ckpts = glob.glob(f'{LOCAL_OUT}/*.pt')
            if ckpts:
                os.makedirs(DRIVE_CKPT, exist_ok=True)
                for f in ckpts:
                    shutil.copy2(f, f'{DRIVE_CKPT}/{os.path.basename(f)}')
                print(f'\n💾 [Auto-save] {len(ckpts)} checkpoint(s) → Drive ✅')
        except Exception as e:
            print(f'\n⚠️  Auto-save failed: {e}')
        time.sleep(interval)

t = threading.Thread(target=auto_save_thread, daemon=True)
t.start()
print(f'🕐 Auto-save: Drive-এ checkpoint every 30 min')

print(f'\n🚀 Training শুরু...')
print(f'   ⚠️  Session শেষ হওয়ার আগে Cell 5 চালাতে ভুলবেন না!\n')
print('=' * 52)

!python train.py config/train_chess_gpt2small.py \
    --init_from={INIT_FROM} \
    --out_dir=out-chess-gpt2small \
    --dtype={dtype} \
    --compile=True

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 5 ── 💾 Checkpoint → Drive
# ⚠️  Session শেষ হওয়ার আগে এটা চালান!
# ═══════════════════════════════════════════════════
import os, shutil, glob

LOCAL_OUT  = '/content/NanoChessGPT-85M/out-chess-gpt2small'
DRIVE_CKPT = '/content/drive/MyDrive/NanoChessGPT/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)

ckpts = glob.glob(f'{LOCAL_OUT}/*.pt')
if not ckpts:
    print('❌ কোনো checkpoint নেই। Cell 4 চালান।')
else:
    print(f'💾 {len(ckpts)} checkpoint(s) Drive-এ save করছি...')
    for f in sorted(ckpts):
        name    = os.path.basename(f)
        dst     = f'{DRIVE_CKPT}/{name}'
        size_mb = os.path.getsize(f) / 1024**2
        print(f'   {name} ({size_mb:.0f} MB)...', end='', flush=True)
        shutil.copy2(f, dst)
        print(' ✅')
    print(f'\n✅ Saved → {DRIVE_CKPT}')
    print(f'\n📋 পরের session:')
    print(f'   Cell 1 → 2 → 3 → 4 চালান → auto-resume হবে!')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 6 ── 📊 Progress Monitor
# Training চলার সময় বা পরে চালান
# ═══════════════════════════════════════════════════
import torch, os, glob

LOCAL_OUT = '/content/NanoChessGPT-85M/out-chess-gpt2small'
ckpts     = sorted(glob.glob(f'{LOCAL_OUT}/*.pt'))

if not ckpts:
    print('এখনো কোনো checkpoint নেই।')
else:
    for path in ckpts:
        ck        = torch.load(path, map_location='cpu')
        it        = ck.get('iter_num', 0)
        val_loss  = ck.get('best_val_loss', float('inf'))
        max_it    = ck.get('config', {}).get('max_iters', 600000)
        pct       = it / max_it * 100
        remain_h  = (max_it - it) * 1.5 / 3600  # ~1.5s/iter with float16+compile
        bar       = '█' * int(pct/5) + '░' * (20 - int(pct/5))

        print(f'\n📊 {os.path.basename(path)}')
        print(f'   [{bar}] {pct:.1f}%')
        print(f'   Iter     : {it:,} / {max_it:,}')
        print(f'   Val loss : {val_loss:.4f}')
        print(f'   Remaining: ~{remain_h:.1f} hours (float16+compile est.)')
        print(f'   Sessions : ~{remain_h/11:.1f} more Colab sessions needed')

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 7 ── 🎮 Generate Chess Moves
# Training শেষে বা চলাকালীন test করুন
# ═══════════════════════════════════════════════════
import os
os.chdir('/content/NanoChessGPT-85M')

# Chess opening দিয়ে শুরু করুন:
START = "e4 e5 Nf3 "  # ← পরিবর্তন করতে পারেন

print(f'Generating from: "{START}"\n')
!python sample.py \
    --out_dir=out-chess-gpt2small \
    --start="{START}" \
    --num_samples=5 \
    --max_new_tokens=300 \
    --temperature=0.8 \
    --top_k=10 \
    --device=cuda